# Implement Integrated Gradients #

## Data Pipeline ##

In [1]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
  print(f'Running on GPU: {torch.cuda.get_device_name(0)}')
else:
  print('Running on CPU')

Running on GPU: NVIDIA GeForce RTX 4070


### Import Dataset ###

In [2]:
import pandas as pd

# path of each of the three csv files of the GoEmotions dataset
# Assuming the files are directly in MyDrive based on the file list
csv_path1 = '../data/goemotions_1.csv'
csv_path2 = '../data/goemotions_2.csv'
csv_path3 = '../data/goemotions_3.csv'

# List of the three files
csv_files = [csv_path1, csv_path2, csv_path3]

# generate DataFrame from each file then concatenate all three DataFrames into one
df = pd.concat((pd.read_csv(filename) for filename in csv_files), ignore_index=True)

#isolate the columns for emotion labels
label_cols = df.columns[9:].tolist()
#print(df.columns)
#print(label_cols)

# turn text column into lists
texts = df['text'].tolist()
# results in a list of lists representing the labels
labels = df[label_cols].values.tolist()

### Get label names ###

In [3]:
label_names = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]
id2label = {i: name for i, name in enumerate(label_names)}

### Split Data into Train and Test ###

In [4]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.1, random_state=42
)

## Tokenization ##

### Load BERT Tokenizer ###

In [5]:
from transformers import AutoTokenizer

MODEL_NAME = 'bert-base-uncased'

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f'Tokenizer type loaded: {type(tokenizer).__name__}')

C:\Users\jinth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer type loaded: BertTokenizer


### Tokenize Train and Test Sets ###

In [6]:
# Tokenize train text
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding='max_length',
    max_length=128,
)

#Tokenize validation text
val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding='max_length',
    max_length=128,
)

print("Tokenization Example (First Text)")
print("Input text:", train_texts[0])
print("Input IDs (Partial):", train_encodings['input_ids'][0][:10], "...")
print("Attention Mask (partial):", train_encodings['attention_mask'][0][:10], '...')

Tokenization Example (First Text)
Input text: the longest week (2014) I don’t know if it’s still available but it was pretty good romantic comedy
Input IDs (Partial): [101, 1996, 6493, 2733, 1006, 2297, 1007, 1045, 2123, 1521] ...
Attention Mask (partial): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] ...


## Create Pytorch Dataset ##

In [7]:
class GoEmotionsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __getitem__(self,idx):
        # converts the encodings lists to Pytorch tensors
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

        # Add multi-hot encoded labels
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

### Instantiate Datasets ###

In [8]:
train_dataset = GoEmotionsDataset(train_encodings, train_labels)
val_dataset = GoEmotionsDataset(val_encodings, val_labels)

### Set up Pytorch DataLoader ###

In [9]:
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

# Set batch size here
BATCH_SIZE = 100

# Training DataLoader
train_dataloader = DataLoader(
    train_dataset,
    # RandomSampler is used to shuffle data during each epoch
    sampler=RandomSampler(train_dataset),
    batch_size=BATCH_SIZE
)

# Validation DataLoader
val_dataloader = DataLoader(
    val_dataset,
    #SequentialSampler is used to ensure the data order is predictable
    sampler=SequentialSampler(val_dataset),
    batch_size=BATCH_SIZE
)

print(f'Train Dataloader created, batch size is {BATCH_SIZE}')
print(f'Validation Dataloader created with a batch size of {BATCH_SIZE}')

Train Dataloader created, batch size is 100
Validation Dataloader created with a batch size of 100


## Define Model ##

In [10]:
from transformers import AutoModelForSequenceClassification, AutoConfig

NUM_LABELS = 28

# configuration for multilabel classification
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config
)

# set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
  print(f"Running on GPU: {torch.cuda.get_device_name(0)}")
else:
  print("Running on CPU (GPU not available or selected)")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 904.54it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

Running on GPU: NVIDIA GeForce RTX 4070


### Import pre-trained model weights ###

In [11]:
weights_path = "../weights/model_weights.pth"
state_dict = torch.load(weights_path, map_location=device)

model.load_state_dict(state_dict)

model.to(device)

model.eval()

C:\Users\jinth\AppData\Local\Temp\ipykernel_44280\689295023.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weights_path, map_location=device)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

## Integrated Gradients ##

In [12]:
import captum
from captum.attr import LayerIntegratedGradients, visualization

## Forward Function ##

In [30]:
# TODO: Needs a forward function: Marian

def forward(input_ids,attention_mask=None):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    return outputs.logits

# instantiate LayerIntegratedGradients
lig = LayerIntegratedGradients(forward, model.bert.embeddings)

# get tensor data
data_iter = iter(val_dataloader)
batch = next(data_iter)

## Generating Baseline ##

In [14]:
# TODO: Generate an empty baseline same size as model tensors: Jin

#getting nessescary tokens
pad_token_id = tokenizer.pad_token_id #for padding empty baseline
sep_token_id = tokenizer.sep_token_id #seperator
cls_token_id = tokenizer.cls_token_id #for start of sentences

#creating baseline
def baseline(input_ids):
    #cloning input ids to ensure it's the same size 
    baseline_ids = input_ids.clone()

    #filling entire tensor with pad tokens
    baseline_ids[:] = pad_token_id

    #putting cls token at beginning
    baseline_ids[:,0] = cls_token_id

    #getting indices for sep tokens
    sep_indices= (input_ids == sep_token_id)
    
    #adding sep tokens to baseline_ids relative to where they were on input_ids
    baseline_ids[sep_indices] = sep_token_id

    return baseline_ids

## Evaluation Loop ##

In [15]:
# Function to calculate the attribution average all the
# embedding elements in the model.
def summarize_attributions(attributions):
    attributions = attributions.sum(dim=-1).squeeze(0)
    attributions = (attributions / torch.norm(attributions))

    return attributions

In [ ]:
# takes the first 10 examples in the batch
for i in range(20):
    # prepare the input one example at a time
    input_ids = batch['input_ids'][i].unsqueeze(0).to(device)
    attention_mask = batch['attention_mask'][i].unsqueeze(0).to(device)

    # get model prediction and label to explain
    logits = forward(input_ids, attention_mask)
    probs = torch.sigmoid(logits)
    top_label_idx = torch.argmax(logits).item()

    # get that label's name 
    top_label_name = id2label[top_label_idx]

    print(f"\nExample {i+1}:  Explaining the top predicted emotion: {top_label_name}")
    print(f"Confidence: {probs[0, top_label_idx]:.2%}")

    baseline_ids = baseline(input_ids)
    #input_embeds = model.bert.embeddings.word_embeddings(input_ids)
    #baseline_embeds = model.bert.embeddings.word_embeddings(baseline_ids)

    # call attribute method to calculate IG
    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids, 
        target=top_label_idx,
        additional_forward_args=(attention_mask),
        n_steps=50,  # we can increase this if all goes well
        return_convergence_delta=True
    )

    # VISUALIZATION HERE - put inside eval loop so each example can be explained
    attributions_sum = summarize_attributions(attributions)
    

    method_eval_score_vis = visualization.VisualizationDataRecord(
        word_attributions = attributions_sum, 
        pred_prob = torch.max(model(input_ids).logits, dim=1)[0].item(),
        pred_class = top_label_name,
        true_class = id2label[torch.argmax(model(input_ids).logits, dim=1)[0].item()],
        attr_class = top_label_name, #get_texts_list[i], # TODO: Fix the mismatch with the text examples! Change the "texts[i]" to something else!
        attr_score = attributions_sum.sum(),
        raw_input_ids = tokenizer.convert_ids_to_tokens(input_ids.squeeze().tolist()),
        convergence_score = delta)
    
    visualization.visualize_text([method_eval_score_vis])


Example 1:  Explaining the top predicted emotion: annoyance
Confidence: 22.75%


In [ ]:
print(attributions_sum)

tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze().tolist())



tensor([ 0.0000,  0.3942, -0.1009,  0.1061, -0.0124,  0.0057, -0.0657,  0.7294,
        -0.1499,  0.0741,  0.0805,  0.0384,  0.0680, -0.0064,  0.0405,  0.0348,
        -0.0409,  0.0549,  0.0393, -0.0404, -0.0125,  0.0079,  0.4567,  0.0338,
        -0.0241, -0.1449,  0.0615,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.00

## **Group Labels with Ekman Groups**

In [17]:
# dictionary of the ekman mapping of the labels and neutral
ekman_mapping = {
    "anger": ["anger", "annoyance", "disapproval"],
    "disgust": ["disgust"],
    "fear": ["fear", "nervousness"],
    "joy": ["joy", "amusement", "approval", "excitement", "gratitude", "love", "optimism", "pride", "relief", "pride"],
    "sadness": ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
    "neutral": ["neutral"]
}

ekman_groups = ["anger", "disgust", "fear", "joy", "sadness", "surprise"] # choose to include "neutral" or not

### **Collapse 28 Label Matrix into 6 Ekman Groups**

In [18]:
import numpy as np

all_labels_matrix = torch.stack([val_dataset[i]['labels'] for i in range(len(val_dataset))]).cpu().numpy()

ekman_matrix = np.zeros((all_labels_matrix.shape[0], len(ekman_groups)))

for i, group in enumerate(ekman_groups):
    target_indices = [k for k, v in id2label.items() if v in ekman_mapping[group]]

    if target_indices: 
        ekman_matrix[:, i] = all_labels_matrix[:, target_indices].max(axis=1)

print(f"Ekman matrix with shape: {ekman_matrix.shape}")

Ekman matrix with shape: (21123, 6)


### **Run Iterative Stratifier**

In [19]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

total_samples = len(val_texts)
train_n = 1000
test_n = total_samples - train_n

msss = MultilabelStratifiedShuffleSplit(n_splits=1, train_size=train_n, test_size=test_n, random_state=42)

for xai_indices, _ in msss.split(val_texts, ekman_matrix):
    sampled_indices = xai_indices

xai_sample_texts = [val_texts[i] for i in sampled_indices]
print(f"Successfully took {len(xai_sample_texts)} samples")

Successfully took 1000 samples


## **Forward Function**

In [ ]:
"""# this loops through each Ekman group to treat it as the 'target' for IG evaluations
# the forward function is in the loop so it is redefined for each label

for group_name in ekman_groups:
    print(f'Evaluating IG for Ekman label: {group_name}')
    
    target_sub_indices = [k for k, v in id2label.items() if v in ekman_mapping[group_name]]

    # This forward function must take the logits from all the labels in an Ekman group and sum them for that group
    def forward_func(inputs):
        mask = (inputs != pad_token_id).long()
        outputs = model(inputs, attention_mask=mask)
        
        return outputs.logits[:, target_sub_indices].sum(dim=1)
    
    lig = LayerIntegratedGradients(forward_func, model.bert.embeddings)

    # here is where to run the attributions for this particular group"""

#getting logits for all labels under fear

group_name = "fear"
print(f'Evaluating IG for Ekman label: {group_name}')

target_sub_indices = [k for k, v in id2label.items() if v in ekman_mapping[group_name]]

# This forward function must take the logits from all the labels in an Ekman group and sum them for that group
def forward_func(inputs, attention_mask):
    mask = (inputs != pad_token_id).long()
    outputs = model(inputs, attention_mask=mask)
        
    return outputs.logits[:, target_sub_indices].sum(dim=1)
    
lig = LayerIntegratedGradients(forward_func, model.bert.embeddings)

#### Collecting Attributions ####

In [38]:
from collections import defaultdict

token_attr_sum = defaultdict(float)
token_count = defaultdict(int)

for i,text in enumerate(xai_sample_texts):

    print(f"Text {i}/1000", end=", ")
    
    text_encodings = tokenizer(text, truncation=True, padding='max_length',max_length=128,return_tensors='pt')

    input_ids = text_encodings['input_ids'].to(device)
    attention_mask = text_encodings['attention_mask'].to(device)

    baseline_ids = baseline(input_ids)

    attributions,delta = lig.attribute(
        inputs = input_ids,
        baselines = baseline_ids,
        additional_forward_args=(attention_mask,),
        n_steps = 50,
        return_convergence_delta=True
    )

    attributions_sum = summarize_attributions(attributions)

    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze().tolist())

    for token,score in zip(tokens,attributions_sum):
        if token not in ["[PAD]","[CLS]","[SEP]"]:
            token_attr_sum[token] += score.item()
            token_count[token] += 1


average_attribution = {token: token_attr_sum[token] / token_count[token] for token in token_attr_sum}

sorted_attribution = sorted(average_attribution.items(), key=lambda x: x[1], reverse = True)

Text 0/1000, Text 1/1000, 

KeyboardInterrupt: 